<a href="https://colab.research.google.com/github/VilasiniRamamoorthy/Mini_Project1/blob/main/Load_Cleaned_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#pip install for pymysql
%pip install pymysql

In [ ]:
#pip install for pandas
%pip install pandas sqlalchemy pymysql

In [ ]:
#Creating cleaned_movies record reading movies.csv after removing null for #budget and revenue used mean value after drop duplicate
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Step 1: Load your CSV
df = pd.read_csv("movies.csv")

# Replace zero budget values with the mean of non-zero budget values
budget_mean = df.loc[df["budget"] != 0, "budget"].mean()
df.loc[df["budget"] == 0, "budget"] = budget_mean

# Replace zero revenue values with the mean of non-zero revenue values
revenue_mean = df.loc[df["revenue"] != 0, "revenue"].mean()
df.loc[df["revenue"] == 0, "revenue"] = revenue_mean

# Step 2: Check the cleaned data
print(df.head())
print(df.isnull().sum())

# Step 3: Enter your TiDB connection details
host = "gateway01.ap-northeast-1.prod.aws.tidbcloud.com"
port = 4000
user = "3k5gspKEptkYxvs.root"
password = "uJVYvVceTGGGT00O"
database = "tmdb_proj_vilasini"

# Step 4: Create the database connection
connection_url = (f"mysql+pymysql://{user}:{password}"
    f"@{host}:{port}/{database}"
    "?ssl_verify_cert=true&ssl_verify_identity=true"
)
engine = create_engine(connection_url, pool_recycle=300)

# Step 5: Upload DataFrame to SQL
df.to_sql(
    name="cleaned_movies",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Data uploaded successfully!")

# Step 6: Verify the uploaded records
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) FROM cleaned_movies")
    )
    print("Total rows uploaded:", result.scalar())

engine.dispose()

In [ ]:
#Creating cleaned_movie_keywords record from reading movie_keywords csv removing duplicates cleaned dataframe
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Step 1: Load your CSV
df = pd.read_csv("movie_keywords.csv")

# Step 2: Check the cleaned data
print(df.head())
print(df.isnull().sum())
print(df.drop_duplicates().sum())

# Step 3: Enter your TiDB connection details
host = "gateway01.ap-northeast-1.prod.aws.tidbcloud.com"
port = 4000
user = "3k5gspKEptkYxvs.root"
password = "uJVYvVceTGGGT00O"
database = "tmdb_proj_vilasini"

# Step 4: Create the database connection
connection_url = (f"mysql+pymysql://{user}:{password}"
    f"@{host}:{port}/{database}"
    "?ssl_verify_cert=true&ssl_verify_identity=true"
)
engine = create_engine(connection_url, pool_recycle=300)

# Step 5: Upload DataFrame to SQL
df.to_sql(
    name="cleaned_movie_keywords",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Data uploaded successfully!")

# Step 6: Verify the uploaded records
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) FROM cleaned_movie_keywords")
    )
    print("Total rows uploaded:", result.scalar())

engine.dispose()

In [ ]:
#Creating cleaned_movie_genres from movie_genres csv after checking for null or duplicates loading cleaned data
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Step 1: Load your CSV
df = pd.read_csv("movie_genres.csv")

# Step 2: Check the cleaned data
print(df.head())
print(df.isnull().sum())
print(df.drop_duplicates().sum())

# Step 3: Enter your TiDB connection details
host = "gateway01.ap-northeast-1.prod.aws.tidbcloud.com"
port = 4000
user = "3k5gspKEptkYxvs.root"
password = "uJVYvVceTGGGT00O"
database = "tmdb_proj_vilasini"

# Step 4: Create the database connection
connection_url = (f"mysql+pymysql://{user}:{password}"
    f"@{host}:{port}/{database}"
    "?ssl_verify_cert=true&ssl_verify_identity=true"
)
engine = create_engine(connection_url, pool_recycle=300)

# Step 5: Upload DataFrame to SQL
df.to_sql(
    name="cleaned_movie_genres",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Data uploaded successfully!")

# Step 6: Verify the uploaded records
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) FROM cleaned_movie_genres")
    )
    print("Total rows uploaded:", result.scalar())

engine.dispose()

In [ ]:
#Created cleaned records genres and crew reading csv - tried loop for 2 file load 2 table creation
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Step 1: Load the CSV files
genres_df = pd.read_csv("genres.csv")
crew_df = pd.read_csv("crew.csv")

# Step 2: Inspect and check the data
for name, df in [("genres", genres_df), ("crew", crew_df)]:
    print(f"\n--- {name.upper()} DATA ---")
    print(df.head())
    print("\nNull values:")
    print(df.isnull().sum())
    print("\nTotal null values:", df.isnull().sum().sum())
    print("\nDuplicate rows:", df.duplicated().sum())
    print("Unique rows:", df.drop_duplicates().shape[0])

# Step 3: Remove duplicate rows (if appropriate)
genres_df = genres_df.drop_duplicates()
crew_df = crew_df.drop_duplicates()

# Step 4: TiDB connection details
host = "gateway01.ap-northeast-1.prod.aws.tidbcloud.com"
port = 4000
user = "3k5gspKEptkYxvs.root"
password = "uJVYvVceTGGGT00O"
database = "tmdb_proj_vilasini"

# Step 5: Create the database connection
connection_url = (
    f"mysql+pymysql://{user}:{password}"
    f"@{host}:{port}/{database}"
    "?ssl_verify_cert=true&ssl_verify_identity=true"
)

engine = create_engine(connection_url, pool_recycle=300)

# Step 6: Upload both DataFrames into separate tables
genres_df.to_sql(
    name="cleaned_genres",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

crew_df.to_sql(
    name="cleaned_crew",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("\nBoth tables uploaded successfully!")

# Step 7: Verify the uploaded row counts
with engine.connect() as conn:
    for table in ["cleaned_genres", "cleaned_crew"]:
        result = conn.execute(
            text(f"SELECT COUNT(*) FROM `{table}`")
        )
        print(f"{table}: {result.scalar()} rows uploaded")

engine.dispose()


In [ ]:
#cast - tried to check unique and duplicates removed
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Step 1: Load your CSV
df = pd.read_csv("cast.csv")

# Step 2: Check the cleaned data
print(df.head())
print(df.isnull().sum())
print("Unique rows:", df.drop_duplicates().shape[0])

# Step 3: Enter your TiDB connection details
host = "gateway01.ap-northeast-1.prod.aws.tidbcloud.com"
port = 4000
user = "3k5gspKEptkYxvs.root"
password = "uJVYvVceTGGGT00O"
database = "tmdb_proj_vilasini"

# Step 4: Create the database connection
connection_url = (f"mysql+pymysql://{user}:{password}"
    f"@{host}:{port}/{database}"
    "?ssl_verify_cert=true&ssl_verify_identity=true"
)
engine = create_engine(connection_url, pool_recycle=300)

# Step 5: Upload DataFrame to SQL
df.to_sql(
    name="cleaned_cast",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Data uploaded successfully!")

# Step 6: Verify the uploaded records
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) FROM cleaned_cast")
    )
    print("Total rows uploaded:", result.scalar())

engine.dispose()